In [1]:
# Runnables Primitive
# Used To connect Task specific Runnable

In [4]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser 

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india").content

# Output Parser
parser = StrOutputParser()

In [7]:
# Most Basic RunnableSquence
from langchain.schema.runnable import RunnableSequence

promt1 = PromptTemplate(
    template= "Write a joke about following \n {topic}",
    input_variables= ['topic']
)

promt2 = PromptTemplate(
    template= "Explain the following joke \n {joke}",
    input_variables= ['joke']
)

chain = RunnableSequence(promt1 , llm_gemini , parser , promt2 , llm_gemini , parser)
chain.invoke("Cricket")

'The joke plays on the double meaning of the phrase "going through the roof."\n\n*   **Literal meaning:** If something is literally going through the roof, you might need a ladder to reach it.\n\n*   **Figurative meaning:** "Going through the roof" is an idiom that means something is increasing dramatically or reaching a very high level (like a price or, in this case, a scoring rate).\n\nThe joke is funny because the cricket fan takes the phrase literally and brings a ladder to the match, implying that he expects to have to climb up to reach the high scoring rate.'

In [11]:
# Runnable Parallel -> Runnable Primitve (each runnable in parallel indepenetly (with same input) and return a dict of outputs)
from langchain.schema.runnable import RunnableParallel

tempalte1 = PromptTemplate(
    template= "Generate a trendy and viral tweet about the following topic : \n {topic} in less then 100 words",
    input_variables= ['topic']
)

tempalte2 = PromptTemplate(
    template= "Generate a trendy and viral linkIn Post about the following topic : \n {topic} in around 200 - 250 words ",
    input_variables= ['topic']
)

final_chain = RunnableParallel({
    'twitter_post' : RunnableSequence(tempalte1 , llm_gemini , parser),
    'LinkeIn_post' : RunnableSequence(tempalte2 , llm_gemini , parser)
})

results = final_chain.invoke("Data Scientist vs Ai Enginner")
print(results['twitter_post'])
print(results['LinkeIn_post'])

Data Scientist: 🕵️‍♂️ Finds insights in data, builds models. AI Engineer: 🛠️ Deploys those models at scale!

Think: Data Scientist is the architect, AI Engineer is the construction crew. Both build the future! 🚀 #DataScience #AI #Tech #Jobs #Career
## Data Scientist vs. AI Engineer: Decoding the Buzz! 🤖🧠

Ever get these two roles mixed up? You're not alone! Data Scientist and AI Engineer are both hot careers, but they tackle different sides of the AI coin.

Think of it this way: **Data Scientists are the visionaries. They *discover* insights from data, building models to predict future trends and answer critical business questions.** They're all about asking "why" and "what."

**AI Engineers are the builders. They *deploy* those models into real-world applications.** They're focused on making the AI work, scale, and integrate seamlessly. Think production-ready AI!

**🔑 Key Differences:**

*   **Data Scientists:** Stats, Machine Learning, Data Visualization, Storytelling
*   **AI Engine

In [12]:
# RunnablePassthrough --> Return exact ouput that it recieve as input
from langchain.schema.runnable import RunnablePassthrough

promt1 = PromptTemplate(
    template= "Write a joke about following \n {topic}",
    input_variables= ['topic']
)

promt2 = PromptTemplate(
    template= "Explain the following joke \n {joke}",
    input_variables= ['joke']
)

chain1 = RunnableSequence(promt1 , llm_gemini , parser)
parallel_chain = RunnableParallel({
    "joke" : RunnablePassthrough(),
    "Joke_Explaination" : RunnableSequence(promt2 , llm_gemini , parser)
})

final_chain = RunnableSequence(chain1 , parallel_chain)
final_chain.invoke("Cricket")

{'joke': 'Why did the cricket fan bring a ladder to the match?\n\nBecause he heard the score was going to be a real **climb**!',
 'Joke_Explaination': 'The joke plays on the double meaning of the word "climb."\n\n*   **Literal Meaning:** A ladder is used to physically climb something.\n*   **Figurative Meaning:** In this context, "climb" refers to a high score in the cricket match. The fan expects the score to increase significantly, hence the "climb."\n\nTherefore, the humor comes from the cricket fan misunderstanding or comically overreacting to the prediction of a high score and bringing a ladder to literally climb the score.'}